# Progetto DIA - A.A 2025/26

Autori: Justin Carideo (justin.carideo@studio.unibo.it), Laura Bertozzi (laura.bertozzi5@studio.unibo.it)

# Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
import seaborn as sns
import kaggle
import os
import os.path
# Librerie scikit-learn
from sklearn.model_selection import KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

Importiamo i dataset presi dal sito https://www.kaggle.com. Questi due dataset fanno riferimento a dati in ambito agricolo, in particolare:
1) **Crop Dataset** -> dataset incentrato su aspetti ambientali per individuare il tipo di coltura dato il tipo di terreno e nutrienti presenti (https://www.kaggle.com/datasets/varshitanalluri/crop-recommendation-dataset)
2) **Fertilizer DataSet** -> dataset incentrato sullo stesso campo ma che vuole individuare il tipo di fertilizzante dato il tipo di terreno e la coltura presente. (https://www.kaggle.com/datasets/nishchalchandel/fertilizer-recommendation)

In [ ]:
CROP_DATASET_ID = "varshitanalluri/crop-recommendation-dataset"
FERTILIZER_DATASET_ID = "nishchalchandel/fertilizer-recommendation"

BASE_DOWNLOAD_DIR = "data"

CROP_DATASET_DIR = os.path.join(BASE_DOWNLOAD_DIR, "dataset_1")
FERTILIZER_DATASET_DIR = os.path.join(BASE_DOWNLOAD_DIR, "dataset_2")

os.makedirs(BASE_DOWNLOAD_DIR, exist_ok=True) # crea la directory se non esiste


def download_kaggle_dataset(dataset_id, dataset_path):
    os.makedirs(dataset_path, exist_ok=True)
    if not os.listdir(dataset_path):
        print(f"Download {dataset_id} in {dataset_path}")
        kaggle.api.dataset_download_files(
            dataset_id,
            path=dataset_path,
            unzip=True
        )
    else:
        print(f"Dataset already in specified path")

download_kaggle_dataset(CROP_DATASET_ID, CROP_DATASET_DIR)
download_kaggle_dataset(FERTILIZER_DATASET_ID, FERTILIZER_DATASET_DIR)

CROP_DATASET_PATH = os.path.join(CROP_DATASET_DIR, "Crop_recommendation.csv")
FERTILIZER_DATASET_PATH = os.path.join(FERTILIZER_DATASET_DIR, "fertilizer_recommendation_dataset.csv")

Creiamo i dataframe di entrambi i dataset:

In [ ]:
crop_df = pd.read_csv(CROP_DATASET_PATH)
fertilizer_df = pd.read_csv(FERTILIZER_DATASET_PATH)

Ora che abbiamo ottenuto i dataset dobbiamo decidere che modello attuare su questi dataset e visto che non hanno indici univoci per riga possiamo attuare una **classificazione** per la predizione di colture e fertilizzanti. In particolare potremmo ragionare con una sorta di *pipeline* dove i dati di entrambi vengono utilizzati per ottenere una classificazione della coltura desiderata date le informazioni relative al terreno e utilizzare i dati ottenuti per calcolare anche il tipo di fertilizzante da attuare.

Questa è una analisi a priori, una volta svolta l'*analisi esplorativa* si potrà procedere con la costruzione del modello.


# Analisi Esplorativa

In [ ]:
crop_df.info()

In [ ]:
fertilizer_df.info()

In [ ]:
crop_df.describe()

In [ ]:
fertilizer_df.describe()

Osserviamo ora quali colture vengono prese in considerazione dai dataset.

Poniamo quindi le stringhe dei nomi delle colture in minuscolo ed eliminiamo eventuali spazi, per poi cercare corrispondenze tra di esse.

In [ ]:
crops_fertilizer = fertilizer_df["Crop"].unique()
crops_fertilizer = [x.lower() and x.strip() for x in crops_fertilizer]
print(crops_fertilizer)

In [ ]:
crops_crop = crop_df["Crop"].unique()
crops_crop = [x.lower() and x.strip() for x in crops_crop]
print(crops_crop)

In [ ]:
common = list(set(crops_fertilizer) & set(crops_crop))
print("Common elements:", common)
difference = list(set(crops_crop) - set(common))
print("Elements in crop but not in fertilizer:", difference)

print("Number of common elements:", len(common))
print("Number of elements in crop:", len(crops_crop))

Notiamo che le colture di `crop_df` rientrano perfettamente nelle colture di `fertilizer_df`. Di conseguenza possiamo portarle allo stesso formato, poi effettuare un join.

In [ ]:
def clean_crop_name(name):
    return str(name).lower().replace(" ", "")

In [ ]:
crop_df["Crop"] = crop_df["Crop"].map(clean_crop_name)

In [ ]:
fertilizer_df["Crop"] = fertilizer_df["Crop"].map(clean_crop_name)

In [ ]:
common_crops = set(fertilizer_df["Crop"]).intersection(set(crop_df["Crop"]))
print(common_crops)
print(f"Common crops = {len(common_crops)}")

### Osservazione dei campioni mediante diagrammi



Innanzitutto analizziamo le variabili prese in considerazione dai due dataset, ponendo particolare attenzione su quelle in comune.

In [ ]:
crop_keys = set(crop_df.keys())
fertilizer_keys = set(fertilizer_df.keys())


print("variabili di \"crop_df\":", list(crop_keys))
print("numero di variabili di \"crop_df\":", len(crop_keys))
print("\nvariabili di \"fertilizer_df\":", list(fertilizer_keys))
print("numero di variabili di \"fertilizer_df\":", len(fertilizer_keys))

common_keys = list(set(crop_keys) & set(fertilizer_keys))
print("\nvariabili comuni:", common_keys)
print("numero di variabili comuni:", len(common_keys))

different_a = list(set(fertilizer_keys) - set(common_keys))
different_b = list(set(crop_keys) - set(common_keys))
different_keys = list(set(crop_keys) ^ set(fertilizer_keys))
print("\nvariabili diverse:", different_keys)
print("numero di variabili diverse:", len(different_keys))

Tra le variabili diverse notiamo che tre di esse differiscono solo nel nome, ossia:
* Humidiy e Moisture
* PH e pH_Value
* Phosphorous e Phosphorus

Le restanti quattro rappresentano parametri effettivamente differenti:
* Soil
* Carbon
* Fertilizer (incognita del fertilizer dataset)
* Remark (che ignoriamo per il momento)

Notiamo come alcuni parametri non facciano riferimento alla stessa scala(i.e. humidity -> in percentuale (72%), Moisture -> in proporzione (0.72)). In compenso, per quanto riguarda il resto:
* Temperature -> °C
* Nitrogen, Phosphorous, Carbon, Potassium -> mg/kg
* pH -> misura adimensionale standard (da 0 a 14)
* Rainfall -> mm

In [ ]:
map_names = {
    "Moisture": "Humidity",
    "PH" : "pH_Value",
    "Phosphorous" : "Phosphorus"
}

fertilizer_df = fertilizer_df.rename(columns=map_names)
fertilizer_df["Humidity"] = fertilizer_df["Humidity"] * 100 # Per evitare problemi con la divisione

Distribuzione delle Colture:

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=crop_df, x='Crop', hue='Crop', palette='viridis', order=crop_df["Crop"].value_counts().index)
plt.title("Distribuzione Colture (crop_df)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=fertilizer_df, x='Crop', hue='Crop', palette='viridis', order=fertilizer_df["Crop"].value_counts().index)
plt.title("Distribuzione Colture (fertilizer_df)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Notiamo come entrambi i dataset presentino 100 campioni per coltura.

Definiamo una funzione per fare il plot di tutti quanti i parametri chimici che andremmo ad analizzare

In [ ]:
def plot_chemicals(chemical):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(crop_df[chemical], kde=True, color='blue', ax=axes[0])
    axes[0].set_title(chemical + ' crop_df', fontsize=12)
    axes[0].set_xlabel(chemical)

    sns.histplot(fertilizer_df[chemical], kde=True, color='red', ax=axes[1])
    axes[1].set_title(chemical + ' fertilizer_df', fontsize=12)
    axes[1].set_xlabel(chemical)

Valori di umidità rilevati:

In [ ]:
plot_chemicals('Humidity')

Temperature misurate:

In [ ]:
plot_chemicals('Temperature')

Precipitazioni misurate: 

In [ ]:
plot_chemicals('Rainfall')

In [ ]:
plot_chemicals('Nitrogen')

In [ ]:
plot_chemicals('Phosphorus')

In [ ]:
plot_chemicals('pH_Value')

In [ ]:
plot_chemicals('Potassium')

Notiamo che c'è una incoerenza con il dataset `fertilizer_df` e che i parametri `Rainfall`, `Potassium`. `Carbon` misurano valori negativi. Essendo una misurazione delle precipitazioni in mm, non è possibile che siano valori negativi. Per tanto abbiamo deciso di mantenere una certa coerenza con i dati e li abbiamo portati a zero:

In [ ]:
fertilizer_df['Potassium'] = fertilizer_df['Potassium'].clip(lower=0)
fertilizer_df['Carbon'] = fertilizer_df['Carbon'].clip(lower=0)
fertilizer_df['Rainfall'] = fertilizer_df['Rainfall'].clip(lower=0)

Questa scelta è stata decisa dal semplice fatto che non è stata trovata documentazione affidabile sulle misurazioni.

### Correlazione tra variabili

Analizziamo la correlazione tra umidità e precipitazioni mediante coefficiente di Pearsonn.

In [ ]:
rice_crop = crop_df[crop_df["Crop"] == "rice"]
rice_fertilizer = fertilizer_df[fertilizer_df["Crop"] == "rice"]
hum_rain_corr_crop = rice_crop["Humidity"].corr(rice_crop["Rainfall"])
hum_rain_corr_fertilizer = rice_fertilizer["Humidity"].corr(rice_fertilizer["Rainfall"])

print(f"Correlation between Humidity and Rainfall in crop_df: {hum_rain_corr_crop:.2f}")
print(f"Correlation between Humidity and Rainfall in fertilizer_df: {hum_rain_corr_fertilizer:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x = rice_crop["Humidity"], y = rice_crop["Rainfall"], color='blue', ax=axes[0])
axes[0].set_title('Humidity vs Rainfall (crop_df)', fontsize=12)
axes[0].set_xlabel('Humidity (%)')
axes[0].set_ylabel('Rainfall (mm)')

sns.scatterplot(x = rice_fertilizer["Humidity"], y = rice_fertilizer["Rainfall"], color='red', ax=axes[1])
axes[1].set_title('Humidity vs Rainfall (fertilizer_df)', fontsize=12)
axes[1].set_xlabel('Humidity (ratio)')
axes[1].set_ylabel('Rainfall (mm)')


Notiamo come non vi sia correlazione tra le due variabili, probabilmente perchè il range dell'umidità è ristretto e perchè la relazione non è lineare. Tentiamo ora la stessa analisi con il coefficiente di Spearman.

In [ ]:
rice_crop = crop_df[crop_df["Crop"] == "rice"]
rice_fertilizer = fertilizer_df[fertilizer_df["Crop"] == "rice"]
hum_rain_corr_crop = rice_crop["Humidity"].corr(rice_crop["Rainfall"], method='spearman')
hum_rain_corr_fertilizer = rice_fertilizer["Humidity"].corr(rice_fertilizer["Rainfall"], method='spearman')

print(f"Correlation between Humidity and Rainfall in crop_df: {hum_rain_corr_crop:.2f}")
print(f"Correlation between Humidity and Rainfall in fertilizer_df: {hum_rain_corr_fertilizer:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x = rice_crop["Humidity"], y = rice_crop["Rainfall"], color='blue', ax=axes[0])
axes[0].set_title('Humidity vs Rainfall (crop_df)', fontsize=12)
axes[0].set_xlabel('Humidity (%)')
axes[0].set_ylabel('Rainfall (mm)')

sns.scatterplot(x = rice_fertilizer["Humidity"], y = rice_fertilizer["Rainfall"], color='red', ax=axes[1])
axes[1].set_title('Humidity vs Rainfall (fertilizer_df)', fontsize=12)
axes[1].set_xlabel('Humidity (ratio)')
axes[1].set_ylabel('Rainfall (mm)')

Il risultato è pressochè identico, confermando che tra le due variabili Humidity/Moisture e Rainfall non c'è correlazione.

### Mappatura variabili ad un formato comune

In [ ]:
column_names = ["Temperature", "Humidity", "Rainfall", "pH_Value", "Nitrogen", "Phosphorus", "Potassium", "Crop"]

merged_df = pd.concat([
    fertilizer_df[column_names],
    crop_df[column_names]],
    axis=0,
    join='outer',
    ignore_index=True
)

merged_df.describe()

# Addestramento dei modelli
Una volta estratte le feature più importanti possiamo passare al tipo di modello che vogliamo addestrare.

## Modello 1
Il primo modello sarà un modello di *classificazione* per la previsione del parametro `Crop` attraverso degli **alberi di classificazione**, in particolare alleniamo due modelli:
- uno che userà `RandomForestClassifier`, che sfrutta l'algoritmo *Random Forest*
- uno che userà `xgboost`, che sfrutta invece *Gradient Boosting*

In [ ]:
y = merged_df['Crop']
X = merged_df.drop('Crop', axis=1)

Definiamo una funzione generale utile per i classificatori. Attualmente stiamo analizzando dati *numerici*, quindi non abbiamo bisogno di One-Hot Encoding o tecniche complesse per elaborare i dati. L'unico strumento di preprocessing incluso è quindi`StandardScaler`.

In [ ]:
def build_scaled_classificator(classifier):
    return Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", classifier)
    ])

### Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [ ]:
# griglia di parametri per Random Forest
param_grid = {
    'classifier__n_estimators': [50, 100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2', None]
}

In [ ]:
rf_scaled = build_scaled_classificator(RandomForestClassifier(random_state=42))

cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rnd_search_rf = RandomizedSearchCV(
    estimator=rf_scaled,
    param_distributions=param_grid,
    n_iter=10,
    cv=cv_stratified,
    scoring="accuracy",
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rnd_search_rf.fit(X_train, y_train)

print(f"\nBest params: {rnd_search_rf.best_params_}")
print(f"Accuracy in CV: {rnd_search_rf.best_score_ * 100:.2f}%")

best_pipeline_model1 = rnd_search_rf.best_estimator_

In [ ]:
def print_model_heatmap(model, cm, title):
    plt.figure(figsize=(16, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=model.classes_,
                yticklabels=model.classes_)
    plt.title(f"Heatmap of {title} data", fontsize=16)
    plt.xlabel(f"{title} - Predicted", fontsize=12)
    plt.ylabel(f"{title} - Actual", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.show()

In [ ]:
y_pred_rf = best_pipeline_model1.predict(X_test)

print("\n--- REPORT (RANDOM FOREST) ---")
print(classification_report(y_test, y_pred_rf))

cm_rf = confusion_matrix(y_test, y_pred_rf, labels=best_pipeline_model1.classes_)

print_model_heatmap(best_pipeline_model1, cm_rf, "Crop")

Riassumendo in una funzione:

In [ ]:
def build_model_with_classifier(model, param_grid, X, y):
    cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rnd_search_m = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=10,
        cv=cv_stratified,
        scoring="accuracy",
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    rnd_search_m.fit(X, y)
    return rnd_search_m

In [ ]:
def train_random_forest(X, y, title) :
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)

    print(f"Accuracy: {accuracy}")

    plt.figure(figsize=(16, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=rf_model.classes_,
                yticklabels=rf_model.classes_)
    plt.title(f"Heatmap con Random Forest - Predizione {title}")
    plt.xlabel(f"{title} predetto")
    plt.ylabel(f"{title} corretto")
    plt.xticks(rotation=45, ha='right')
    plt.show()

    return rf_model

In [ ]:
def BestParams(n_estimators, max_depth, min_samples_split, min_samples_leaf):
    # Parametri da esplorare
    param_grid = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf
    }

    rf_base = RandomForestClassifier(random_state=42)

    validation_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rf_random = RandomizedSearchCV(
        estimator=rf_base,
        param_distributions=param_grid,
        n_iter=20,
        cv=validation_stratified,
        scoring='accuracy',
        verbose=2,
        random_state=42,
        n_jobs=-1
    )

    rf_random.fit(X, y)

    return rf_random.best_params_, rf_random.best_score_

In [ ]:
bestParams, bestScore = BestParams([50, 100, 200, 300], [None, 10, 20, 30, 40], [2, 5, 10], [1, 2, 4])
print(f"Best parameters: {bestParams}")
print(f"Best score: {bestScore * 100:.2f}%")

### XGBoost
Laura

### SVM

Laura

### Logistic Regression


In [ ]:
lr_scaled = build_scaled_classificator(LogisticRegression(random_state=42, max_iter=2000))

param_grid_lr = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__solver': ['lbfgs', 'saga'],
    'classifier__class_weight': [None, 'balanced']
}

rnd_search_lr = build_model_with_classifier(lr_scaled, param_grid_lr, X_train, y_train)

print(f"\nBest params: {rnd_search_lr.best_params_}")
print(f"Accuracy in CV: {rnd_search_lr.best_score_ * 100:.2f}%")

best_pipeline_lr = rnd_search_lr.best_estimator_

In [ ]:
y_pred_lr = best_pipeline_lr.predict(X_test)
cm_lr = confusion_matrix(y_test, y_pred_lr, labels=best_pipeline_lr.classes_)
print_model_heatmap(best_pipeline_lr, cm_lr, "Crop")

## Modello 2

Questo modello dovrà predire la feature `fertilizer` sfruttando il dataset `fertilizer_df`, una volta effettuate alcune necessarie modifiche. 
(da togliere colonna remark)

### Preparazione dei dati

Il secondo modello sfrutterà anche le colonne del dataset che presentano dati di tipo **categorico**. Di conseguenza bisogna ragionare sulla codifica di questi dati in modo da poterli inserire nel modello di predizione. L'approccio standard, seguito anche da noi è quello dell'**One-Hot Encoding**, attraverso il quale il dato viene codificato sfruttando un numero di colonne pari alle categorie, delle quali solo quella corrispondente alla variabile in analisi presenterà valore 1. Per esempio, se nella fase precedente abbiamo trovato che una riga presenta la coltura `rice` allora nel dataset che dovremmo esaminare, tale riga avrà, in corrispondenza della colonna `Crop_rice`, il valore 1, mentre per le altre avrà 0.

In [ ]:
def cat_num(df):
    categorical = df.select_dtypes(include=['object', 'str']).columns
    numeric = df.select_dtypes(include=['number']).columns
    return categorical, numeric

In [ ]:
#fertilizer_df senza colonna "Remark"
fertilizer_df = fertilizer_df.drop(columns=["Remark"])
fertilizer_df_data = fertilizer_df.drop(columns=["Fertilizer"])
fertilizer_column = fertilizer_df["Fertilizer"]

# Divisione tra colonne categoriche e numeriche
categorical, numeric = cat_num(fertilizer_df_data)


In [ ]:
print("fertilizer_df columns:")
print("categorical:\t",categorical)
print("numeric:\t",numeric)

In [ ]:
def build_scaled_encoded_preprocessor(numeric, categorical):
    return ColumnTransformer([
        ("numeric", StandardScaler(), numeric ),
        ("categorical", OneHotEncoder(drop='first', sparse_output=False), categorical)
    ])

In [ ]:
def build_scaled_encoded_classificator(classifier, numeric, categorical):
    return Pipeline([
        ("preprocessor", build_scaled_encoded_preprocessor(numeric, categorical)),
        ("classifier", classifier)
    ])

In [ ]:
rf_scaled_encoded = build_scaled_encoded_classificator(RandomForestClassifier(random_state=42), numeric, categorical)

model = build_model_with_classifier(rf_scaled_encoded, param_grid, fertilizer_df_data, fertilizer_column)

### Random Forest

In [ ]:

print(f"\nBest params: {model.best_params_}")
print(f"Accuracy in CV: {model.best_score_ * 100:.2f}%")

best_pipeline_lr = model.best_estimator_

### XGBoost


In [ ]:
fert_labels = LabelEncoder()
y_encoded = fert_labels.fit_transform(y)

In [ ]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
])

pipeline_xgb = Pipeline(steps=[
    ("preproc", preprocessor),
    ("classifier", XGBClassifier(random_state=42, eval_metric="mlogloss", n_jobs=-1))
])

param_grid_xgb = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "classifier__max_depth": [3, 5, 7, 9],
    "classifier__subsample": [0.8, 0.9, 1.0]
}

rnd_search_xgb_2 = build_model_with_classifier(pipeline_xgb, param_grid_xgb, X_train_f, y_train_f)

print(f"\nBest params: {rnd_search_xgb_2.best_params_}")
print(f"Accuracy in CV: {rnd_search_xgb_2.best_score_ * 100:.2f}%")

best_pipeline_model2 = rnd_search_xgb_2.best_estimator_

# Interfaccia Web

In [ ]:
import joblib

rf_crop_model_name = 'rf-crop-model.pkl'
# joblib.dump(rf_model, rf_crop_model_name)